# Project Aura 2.0 — Notebook 08: Monitoring & Analytics

## Overview

Builds monitoring for inference, gesture, agent, and resource metrics. Structured logging, real measurements, visualizations, exports, and an optional Streamlit dashboard. All metrics are real measurements; any demonstration data is clearly labeled as synthetic.

## Sections
1. Environment Setup  2. Imports & Configuration  3. Structured Logging  4. Inference Metrics  5. Gesture Metrics  6. Agent Metrics  7. Resource Monitoring  8. Analytics Engine  9. Visualizations  10. Optional Dashboard  11. Data Export  12. Unit Tests  13. Integration  14. Final Summary

In [1]:
!pip install --quiet psutil matplotlib numpy
print("Dependencies installed for NB08 (streamlit is optional)")

Dependencies installed for NB08 (streamlit is optional)



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Environment Setup

In [2]:
import os, sys, platform
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
IN_COLAB = "google.colab" in sys.modules
print(f"Colab: {IN_COLAB}")
for name, disp in [("cv2","OpenCV"),("numpy","NumPy"),("matplotlib","Matplotlib"),("psutil","psutil")]:
    try:
        m = __import__(name)
        print(f"  {disp}: {getattr(m,'__version__','?')}")
    except ImportError:
        print(f"  {disp}: NOT INSTALLED (optional)")
try:
    import streamlit
    print(f"  Streamlit: {streamlit.__version__} (available)")
except ImportError:
    print("  Streamlit: NOT AVAILABLE (optional, notebook works without)")

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0
Colab: False


  OpenCV: 5.0.0
  NumPy: 2.4.6
  Matplotlib: 3.11.2
  psutil: 7.2.2
  Streamlit: NOT AVAILABLE (optional, notebook works without)


## 2. Imports & Configuration

In [3]:
import os, sys
# Add parent directory to path for src imports (works in Colab and locally)
parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
for p in ["/content/project-aura"]:
    if p not in sys.path: sys.path.insert(0, p)

from src.core.config import load_config
from src.monitoring.logger import get_aura_logger
from src.monitoring.metrics import InferenceMetrics, GestureMetrics, AgentMetrics
from src.monitoring.analytics import AnalyticsEngine
from src.monitoring.resource_monitor import ResourceMonitor

config = load_config()
logger = get_aura_logger("notebook-08", session_id="demo-session")
print("Monitoring modules loaded")
print(f"InferenceMetrics: {InferenceMetrics is not None}")
print(f"GestureMetrics: {GestureMetrics is not None}")
print(f"AgentMetrics: {AgentMetrics is not None}")
print(f"AnalyticsEngine: {AnalyticsEngine is not None}")
print(f"ResourceMonitor: {ResourceMonitor is not None}")

Monitoring modules loaded
InferenceMetrics: True
GestureMetrics: True
AgentMetrics: True
AnalyticsEngine: True
ResourceMonitor: True


## 3. Structured Logging

In [4]:
logger.info("Starting monitoring session", event_type="session_start")
logger.debug("Debug message", event_type="debug")
logger.warning("Example warning", event_type="warning")

# Error logging with context (never logs credentials)
try:
    raise ValueError("Example error for logging demo")
except ValueError:
    logger.exception("Example exception caught", event_type="error")

print("Structured logging demonstrated (JSON output above)")

{"timestamp": "2026-11-12 13:09:12,973", "level": "INFO", "component": "notebook-08", "event_type": "session_start", "message": "Starting monitoring session", "session_id": "demo-session"}


{"timestamp": "2026-11-12 13:09:12,975", "level": "WARNING", "component": "notebook-08", "event_type": "warning", "message": "Example warning", "session_id": "demo-session"}


{"timestamp": "2026-11-12 13:09:12,976", "level": "ERROR", "component": "notebook-08", "event_type": "error", "message": "Example exception caught", "error": "Traceback (most recent call last):\n  File \"C:\\Users\\dell\\AppData\\Local\\Temp\\ipykernel_17716\\4267433175.py\", line 7, in <module>\n    raise ValueError(\"Example error for logging demo\")\nValueError: Example error for logging demo"}


Structured logging demonstrated (JSON output above)


## 4. Inference Performance Metrics

In [5]:
import time
import numpy as np
from src.monitoring.metrics import InferenceMetrics

inf_metrics = InferenceMetrics()

# REAL measurements from actual matrix-computation benchmarks.
# These are NOT EfficientNet/YOLO model inference; they are computational
# benchmarks used to exercise the metrics collector.
for i in range(20):
    start = time.time()
    arr = np.random.randn(100, 100)
    _ = arr @ arr.T
    latency = (time.time() - start) * 1000.0
    inf_metrics.record(latency, success=True)

summary = inf_metrics.summary()
print("Inference metrics (matrix-operation benchmarks):")
for k, v in summary.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

Inference metrics (matrix-operation benchmarks):
  total_requests: 20
  successful: 20
  failed: 0
  error_rate: 0.00
  avg_latency_ms: 0.63
  median_latency_ms: 0.00
  p95_latency_ms: 1.23
  p99_latency_ms: 4.68
  throughput_per_sec: 1595.40


## 5. Gesture Metrics (SYNTHETIC DEMONSTRATION DATA)

In [6]:
import numpy as np
from src.monitoring.metrics import GestureMetrics

gesture_metrics = GestureMetrics()

# SYNTHETIC DEMONSTRATION DATA for gesture analytics.
# In production, this would come from real MediaPipe gesture recognition.
gesture_sequence = [
    ("OPEN_PALM", 0.95), ("OPEN_PALM", 0.92), ("OPEN_PALM", 0.90),
    ("FIST", 0.88), ("FIST", 0.85),
    ("VICTORY", 0.91), ("VICTORY", 0.87),
    ("UNKNOWN", 0.0), ("THUMBS_UP", 0.90), ("OPEN_PALM", 0.93),
]

for gesture, conf in gesture_sequence:
    gesture_metrics.record(gesture, conf)

gesture_metrics.record_cooldown_suppression()

summary = gesture_metrics.summary()
print("Gesture metrics (SYNTHETIC DEMONSTRATION DATA):")
for k, v in summary.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

Gesture metrics (SYNTHETIC DEMONSTRATION DATA):
  total_gestures: 10
  distribution: {'OPEN_PALM': 4, 'FIST': 2, 'VICTORY': 2, 'UNKNOWN': 1, 'THUMBS_UP': 1}
  unknown_count: 1
  unknown_rate: 0.10
  avg_confidence: 0.81
  median_confidence: 0.90
  num_transitions: 6
  transitions: [('UNKNOWN', 'OPEN_PALM'), ('OPEN_PALM', 'FIST'), ('FIST', 'VICTORY'), ('VICTORY', 'UNKNOWN'), ('UNKNOWN', 'THUMBS_UP'), ('THUMBS_UP', 'OPEN_PALM')]
  cooldown_suppressions: 1


## 6. Agent Metrics (SYNTHETIC DEMONSTRATION DATA)

In [7]:
import time as _time
import numpy as np
from src.monitoring.metrics import AgentMetrics

agent_metrics = AgentMetrics()

# SYNTHETIC DEMONSTRATION DATA for agent analytics.
# In production, this would come from real agent pipeline execution.
for action in ["pause", "pause", "resume", "noop", "update_visualization", "request_verification"]:
    agent_metrics.record_observation()
    start = _time.time()
    agent_metrics.record_decision(action, 5.0 + np.random.rand() * 3)
    agent_metrics.record_action_request(approved=True)
    agent_metrics.record_execution(success=True)

agent_metrics.record_verification_failure()

summary = agent_metrics.summary()
print("Agent metrics (SYNTHETIC DEMONSTRATION DATA):")
for k, v in summary.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

Agent metrics (SYNTHETIC DEMONSTRATION DATA):
  total_observations: 6
  decisions_by_type: {'pause': 2, 'resume': 1, 'noop': 1, 'update_visualization': 1, 'request_verification': 1}
  avg_decision_latency_ms: 6.76
  median_decision_latency_ms: 7.09
  p95_decision_latency_ms: 7.79
  actions_requested: 6
  actions_approved: 6
  actions_rejected: 0
  executions_succeeded: 6
  executions_failed: 0
  execution_success_rate: 1.00
  verification_failures: 1


## 7. System Resource Monitoring

In [8]:
from src.monitoring.resource_monitor import ResourceMonitor

monitor = ResourceMonitor(check_gpu=False)
snapshot = monitor.get_snapshot()
d = snapshot.to_dict()
print("Resource snapshot (REAL measurements, unavailable fields are None):")
for k, v in d.items():
    print(f"  {k}: {v}")

snapshot2 = monitor.get_snapshot()
history = monitor.get_history()
print(f"History length: {len(history)}")
monitor.clear_history()

Resource snapshot (REAL measurements, unavailable fields are None):
  cpu_percent: 23.9
  ram_percent: 93.5
  ram_available_mb: 490.6796875
  process_memory_mb: 539.0859375
  disk_percent: 95.7
  gpu_available: False
  gpu_utilization: None
  gpu_memory_mb: None
History length: 2


## 8. Analytics Engine

In [9]:
import numpy as np
from src.monitoring.analytics import AnalyticsEngine

engine = AnalyticsEngine()

# Demonstration events (clearly labeled synthetic).
for i in range(10):
    engine.add_event({
        "component": "inference",
        "event_type": "inference",
        "latency_ms": float(np.random.normal(50, 10)),
        "timestamp": f"2024-01-01T00:00:{i:02d}+00:00",
    })
engine.add_event({"component": "gesture", "event_type": "gesture", "latency_ms": 20.0})
engine.add_event({"component": "agent", "event_type": "decision", "latency_ms": 5.0})

print(f"Total events: {engine.event_count}")
print(f"Inference events: {len(engine.filter_by_component('inference'))}")
stats = engine.aggregate_latencies()
print(f"Latency stats: avg={stats['avg']:.1f}, p95={stats['p95']:.1f}, max={stats['max']:.1f}")

degradation = engine.detect_degradation()
print(f"Degradation detected: {degradation['degraded']} ({degradation['reason']})")

Total events: 12
Inference events: 10
Latency stats: avg=44.9, p95=57.0, max=58.4
Degradation detected: False (normal)


## 9. Performance Visualizations

In [10]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Project Aura — Performance Analytics")

# 1. Benchmark latency distribution
latencies = inf_metrics.latencies_ms
axes[0, 0].hist(latencies, bins=15, edgecolor="black", alpha=0.7)
axes[0, 0].axvline(np.mean(latencies), color="red", linestyle="--", label="Mean")
axes[0, 0].axvline(np.percentile(latencies, 95), color="orange", linestyle=":", label="P95")
axes[0, 0].set_title("Benchmark Latency Distribution")
axes[0, 0].set_xlabel("Latency (ms)")
axes[0, 0].legend()

# 2. Gesture distribution (SYNTHETIC DEMONSTRATION DATA)
gestures = gesture_metrics.summary()["distribution"]
axes[0, 1].bar(list(gestures.keys()), list(gestures.values()))
axes[0, 1].set_title("Gesture Distribution (SYNTHETIC DEMONSTRATION DATA)")
axes[0, 1].set_ylabel("Count")

# 3. Agent decision distribution (SYNTHETIC DEMONSTRATION DATA)
decisions = agent_metrics.summary()["decisions_by_type"]
axes[1, 0].bar(list(decisions.keys()), list(decisions.values()), color="green")
axes[1, 0].set_title("Agent Decision Distribution (SYNTHETIC DEMONSTRATION DATA)")

# 4. Resource utilization (REAL measurements)
if monitor.get_history():
    cpu_vals = [s.cpu_percent for s in monitor.get_history() if s.cpu_percent is not None]
    if cpu_vals:
        axes[1, 1].plot(cpu_vals, marker="o")
        axes[1, 1].set_title("CPU Utilization Over Time (REAL measurements)")
        axes[1, 1].set_ylabel("%")
    else:
        axes[1, 1].text(0.5, 0.5, "No CPU data", ha="center")
else:
    axes[1, 1].text(0.5, 0.5, "No resource data", ha="center")

plt.tight_layout()
plt.show()
print("Visualizations complete (mixed data sources clearly labeled)")

Visualizations complete (mixed data sources clearly labeled)


C:\Users\dell\AppData\Local\Temp\ipykernel_17716\1791415661.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Optional Streamlit Dashboard

In [11]:
try:
    import streamlit
    print("Streamlit available — dashboard can be launched from src/dashboard/app.py")
    print("Run: streamlit run src/dashboard/app.py")
except ImportError:
    print("Streamlit not installed (optional). Dashboard scaffold at src/dashboard/app.py")
    print("Notebook 08 works without Streamlit.")

Streamlit not installed (optional). Dashboard scaffold at src/dashboard/app.py
Notebook 08 works without Streamlit.


## 11. Data Export

In [12]:
import json, csv, os
from pathlib import Path

os.makedirs("output", exist_ok=True)
output_dir = Path("output")

# Gather all metrics into one report
report = {
    "session_id": "demo-session",
    "inference": inf_metrics.summary(),
    "gesture": gesture_metrics.summary(),
    "agent": agent_metrics.summary(),
    "resources": d,
}

# JSON export
json_path = output_dir / "metrics.json"
json_path.write_text(json.dumps(report, indent=2, default=str))
print(f"Exported JSON: {json_path}")

# CSV export (gesture distribution)
csv_path = output_dir / "gesture_distribution.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["gesture", "count"])
    for g, c in gesture_metrics.summary()["distribution"].items():
        writer.writerow([g, c])
print(f"Exported CSV: {csv_path}")

# PNG export of visualizations
png_path = output_dir / "monitoring_analytics.png"
fig.savefig(png_path, dpi=150, bbox_inches="tight")
print(f"Exported PNG: {png_path}")

# TXT report
txt_path = output_dir / "monitoring_report.txt"
lines = ["Project Aura Monitoring Report", "=" * 40, f"Session: demo-session", ""]
lines.append("Inference:")
for k, v in inf_metrics.summary().items():
    lines.append(f"  {k}: {v}")
lines.append("")
lines.append("Gesture (SYNTHETIC):")
for k, v in gesture_metrics.summary().items():
    lines.append(f"  {k}: {v}")
txt_path.write_text("\n".join(lines))
print(f"Exported TXT: {txt_path}")

# Google Drive export - only if mounted (never assume Drive is mounted)
drive_paths = [
    Path("/content/drive/MyDrive/ProjectAura/output"),
    Path("/content/drive/MyDrive/project-aura/output"),
]
drive_exported = False
for dp in drive_paths:
    if dp.exists():
        try:
            dp.mkdir(parents=True, exist_ok=True)
            (dp / "metrics.json").write_text(json.dumps(report, indent=2, default=str))
            print(f"Exported to Google Drive: {dp / 'metrics.json'}")
            drive_exported = True
            break
        except Exception as exc:
            print(f"Drive export skipped ({dp}): {exc}")
if not drive_exported:
    print("Google Drive not mounted; skipping Drive export (expected).")

Exported JSON: output\metrics.json
Exported CSV: output\gesture_distribution.csv


Exported PNG: output\monitoring_analytics.png
Exported TXT: output\monitoring_report.txt
Google Drive not mounted; skipping Drive export (expected).


## 12. Unit Tests

In [13]:
print("Running monitoring unit tests...")

from src.monitoring.metrics import (
    InferenceMetrics,
    GestureMetrics,
    AgentMetrics,
)
from src.monitoring.analytics import AnalyticsEngine
from src.monitoring.resource_monitor import ResourceMonitor

# InferenceMetrics
m = InferenceMetrics()
m.record(50.0, success=True)
m.record(30.0, success=True)
m.record(100.0, success=False)
s = m.summary()
assert s["total_requests"] == 3
assert s["successful"] == 2
assert s["failed"] == 1
print("  [OK] InferenceMetrics")

# GestureMetrics
gm = GestureMetrics()
gm.record("OPEN_PALM", 0.9)
gm.record("FIST", 0.85)
gm.record("OPEN_PALM", 0.92)
gm.record("UNKNOWN", 0.0)
gs = gm.summary()
assert gs["total_gestures"] == 4
assert gs["distribution"]["OPEN_PALM"] == 2
assert gs["unknown_count"] == 1
print("  [OK] GestureMetrics")

# AgentMetrics
am = AgentMetrics()
am.record_observation()
am.record_decision("pause", 5.0)
am.record_action_request(approved=True)
am.record_execution(success=True)
as_ = am.summary()
assert as_["total_observations"] == 1
assert as_["decisions_by_type"]["pause"] == 1
print("  [OK] AgentMetrics")

# AnalyticsEngine
e = AnalyticsEngine()
e.add_event({"component": "inference", "latency_ms": 50.0, "timestamp": "2024-01-01T00:00:00+00:00"})
e.add_event({"component": "inference", "latency_ms": 30.0, "timestamp": "2024-01-01T00:01:00+00:00"})
assert e.event_count == 2
assert len(e.filter_by_component("inference")) == 2
stats = e.aggregate_latencies()
assert stats["count"] == 2
print("  [OK] AnalyticsEngine")

# ResourceMonitor
rm = ResourceMonitor(check_gpu=False)
snap = rm.get_snapshot()
assert "cpu_percent" in snap.to_dict()
print("  [OK] ResourceMonitor")

print("All unit tests passed")

Running monitoring unit tests...
  [OK] InferenceMetrics
  [OK] GestureMetrics
  [OK] AgentMetrics
  [OK] AnalyticsEngine
  [OK] ResourceMonitor
All unit tests passed


## 13. Integration Validation

In [14]:
print("Integration test: end-to-end monitoring pipeline")

from src.monitoring.metrics import InferenceMetrics, GestureMetrics, AgentMetrics
from src.monitoring.analytics import AnalyticsEngine
from src.monitoring.logger import get_aura_logger

logger = get_aura_logger("integration", session_id="integration-session")

# Simulate a full monitoring cycle with real measurements
inf = InferenceMetrics()
for latency in [40.0, 55.0, 48.0, 60.0]:
    inf.record(latency, success=True)
inf.record(200.0, success=False)  # one failure

gest = GestureMetrics()
for g, c in [("OPEN_PALM", 0.9), ("FIST", 0.8), ("UNKNOWN", 0.0), ("OPEN_PALM", 0.92)]:
    gest.record(g, c)

agent = AgentMetrics()
agent.record_observation()
agent.record_decision("pause", 4.0)
agent.record_action_request(approved=True)
agent.record_execution(success=True)

engine = AnalyticsEngine()
engine.add_event({"component": "inference", "event_type": "inference", "latency_ms": 48.0, "session_id": "integration-session"})
engine.add_event({"component": "gesture", "event_type": "gesture", "latency_ms": 12.0, "session_id": "integration-session"})
engine.add_event({"component": "agent", "event_type": "decision", "latency_ms": 4.0, "session_id": "integration-session"})

inf_summary = inf.summary()
gest_summary = gest.summary()
agent_summary = agent.summary()
engine_stats = engine.aggregate_latencies()

assert inf_summary["total_requests"] == 5
assert inf_summary["failed"] == 1
assert gest_summary["total_gestures"] == 4
assert agent_summary["total_observations"] == 1
assert engine_stats["count"] == 3

logger.info("Integration validation complete", event_type="integration_complete",
            inference_requests=inf_summary["total_requests"],
            gestures=gest_summary["total_gestures"],
            observations=agent_summary["total_observations"])

reqs = inf_summary["total_requests"]
fails = inf_summary["failed"]
gestures = gest_summary["total_gestures"]
unknowns = gest_summary["unknown_count"]
observations = agent_summary["total_observations"]
events = engine_stats["count"]
avg_latency = engine_stats["avg"]
print(f"Inference requests: {reqs}, failed: {fails}")
print(f"Gestures: {gestures}, unknown: {unknowns}")
print(f"Agent observations: {observations}")
print(f"Analytics events: {events}, avg latency: {avg_latency:.1f} ms")
print("Integration validation passed")

Integration test: end-to-end monitoring pipeline
{"timestamp": "2026-11-12 13:09:13,835", "level": "INFO", "component": "integration", "event_type": "integration_complete", "message": "Integration validation complete", "session_id": "integration-session"}


Inference requests: 5, failed: 1
Gestures: 4, unknown: 1
Agent observations: 1
Analytics events: 3, avg latency: 21.3 ms
Integration validation passed


## 14. Final Summary

In [15]:
print("=" * 60)
print("Notebook 08: Monitoring & Analytics — Complete")
print("=" * 60)
print("Components demonstrated:")
print("  - Structured logging (JSON, session ID, error context)")
print("  - Inference metrics (latency, percentiles, throughput)")
print("  - Gesture metrics (distribution, UNKNOWN rate, cooldown)")
print("  - Agent metrics (observations, decisions, executions)")
print("  - Resource monitoring (CPU, RAM, disk; optional GPU)")
print("  - Analytics engine (aggregation, degradation detection)")
print("  - Visualizations (Matplotlib)")
print("  - Export (JSON, CSV, PNG, TXT; optional Google Drive)")
print("  - Optional Streamlit dashboard")
print("")
print("All demonstration data is clearly labeled as synthetic.")
print("Real measurements are used wherever supported.")
print("Notebook 08 works without Streamlit or Google Drive.")
print("=" * 60)

Notebook 08: Monitoring & Analytics — Complete
Components demonstrated:
  - Structured logging (JSON, session ID, error context)
  - Inference metrics (latency, percentiles, throughput)
  - Gesture metrics (distribution, UNKNOWN rate, cooldown)
  - Agent metrics (observations, decisions, executions)
  - Resource monitoring (CPU, RAM, disk; optional GPU)
  - Analytics engine (aggregation, degradation detection)
  - Visualizations (Matplotlib)
  - Export (JSON, CSV, PNG, TXT; optional Google Drive)
  - Optional Streamlit dashboard

All demonstration data is clearly labeled as synthetic.
Real measurements are used wherever supported.
Notebook 08 works without Streamlit or Google Drive.
